# Data Cleaning and Processing — PT Kirimin
Dataset sengaja kotor: duplicate order, kota bervariasi, tanggal campur, dan nilai tidak valid.

> Diverifikasi: Python v3.14.7 — sumber: https://docs.python.org/3/ — tanggal cek: 2026-09-18
> Diverifikasi: pandas v3.0.3 — sumber: https://pandas.pydata.org/docs/whatsnew/v3.0.3.html — tanggal cek: 2026-09-18

In [ ]:
import pandas as pd
raw = pd.DataFrame({
 'order_id':['O-1','O-2','O-2',None,'O-5'],
 'origin_city':[' Jakarta','JKT','JKT','Bandung','jakarta'],
 'order_created_at':['01/09/2026','2026-09-02','2026-09-02','2026-09-03','2026-09-04'],
 'order_value_idr':['200000','350000','350000','500000','999999999']
})
profile = pd.DataFrame({'dtype':raw.dtypes.astype(str), 'nulls':raw.isna().sum(), 'unique':raw.nunique()})
profile

In [ ]:
clean = raw.copy()
clean['origin_city'] = clean['origin_city'].astype('string').str.strip().str.lower().replace({'jkt':'jakarta'}).str.title()
clean['order_created_at'] = pd.to_datetime(clean['order_created_at'], dayfirst=True, errors='coerce')
clean['order_value_idr'] = pd.to_numeric(clean['order_value_idr'], errors='coerce')
invalid = clean['order_id'].isna() | clean['order_value_idr'].isna() | (clean['order_value_idr'] > 10_000_000)
quarantine = clean.loc[invalid].assign(rejection_reason='missing key, invalid value, or domain outlier')
curated = clean.loc[~invalid].drop_duplicates('order_id', keep='last').reset_index(drop=True)
curated, quarantine

## Mini-exercise
1. Tambahkan `source_batch_id` dan `processed_at`.
2. Buat mapping kota yang dapat diaudit dan uji bahwa pipeline idempotent.
# TODO: tulis eksperimen Anda.

## Takeaway
Cleaning yang baik menghasilkan data curated sekaligus quarantine dan alasan keputusan.